In [48]:
import pandas as pd

In [49]:
from datasets import load_dataset
ds = load_dataset("AgentPublic/cnil")

# Etape 1 : Exploration de la data

In [50]:
print(ds)
df = ds["train"].to_pandas()
print(df.head(5))
print(df["text"][0])

DatasetDict({
    train: Dataset({
        features: ['chunk_id', 'doc_id', 'chunk_index', 'chunk_xxh64', 'nature', 'status', 'nature_delib', 'title', 'full_title', 'number', 'date', 'text', 'chunk_text', 'embeddings_bge-m3'],
        num_rows: 64667
    })
})
                 chunk_id                doc_id  chunk_index  \
0  CNILTEXT000017651752_1  CNILTEXT000017651752            1   
1  CNILTEXT000017651752_2  CNILTEXT000017651752            2   
2  CNILTEXT000017651752_3  CNILTEXT000017651752            3   
3  CNILTEXT000017651752_4  CNILTEXT000017651752            4   
4  CNILTEXT000017651752_5  CNILTEXT000017651752            5   

        chunk_xxh64        nature   status        nature_delib  \
0  49574288592aa3f7  DELIBERATION  VIGUEUR  Autre autorisation   
1  f7b2925dd0cb489e  DELIBERATION  VIGUEUR  Autre autorisation   
2  656d16a4f33193a8  DELIBERATION  VIGUEUR  Autre autorisation   
3  683b5018adea0ad7  DELIBERATION  VIGUEUR  Autre autorisation   
4  6c6e72bc66fddc96  DEL

In [51]:
# Combien de décisions
print("df[doc_id].nunique() : ", df["doc_id"].nunique())

df[doc_id].nunique() :  9983


In [52]:
# Nature des sanctions 
print("sanctions :", df["nature"].value_counts())
print("sanctions :", df["nature_delib"].value_counts())

sanctions : nature
DELIBERATION    59768
DECISION         4899
Name: count, dtype: int64
sanctions : nature_delib
Autre autorisation                   23529
Avis                                 21909
Sanction                              6391
Référentiel/Règlement type/Norme      3738
Autorisation de recherche             2717
Autorisation de transfert             2020
Mise en demeure                       1145
Recommandation/Lignes directrices     1045
Disposition interne CNIL               951
Certification/Label                    651
Autorisation d'évaluation              138
Clôture                                118
Autre décision                          26
AVIS                                     2
Name: count, dtype: int64


In [53]:
# Types 
df.dtypes

chunk_id               str
doc_id                 str
chunk_index          int32
chunk_xxh64            str
nature                 str
status                 str
nature_delib           str
title                  str
full_title             str
number                 str
date                   str
text                   str
chunk_text             str
embeddings_bge-m3      str
dtype: object

In [54]:

# Convert date to datetime 
df["date"] = pd.to_datetime(df["date"])

# Remove leading/trailing whitespace in the number and full_title columns
df["number"] = df["number"].str.strip()
df["full_title"] = df["full_title"].str.strip()

# We will only consider decisions with a sanction or a mise en demeure (nature_delib) for the graph
df_sanctions = df[(df["nature_delib"].isin(["Mise en demeure","Sanction"])) & (df["date"] >= "2018-05-25")].copy()
print("Number of sanctions and mise en demeure:", df_sanctions["doc_id"].nunique())

# Min and max date
print("Minimal date :", df_sanctions["date"].min())
print("Maximal date :", df_sanctions["date"].max())

# Adding the column year to the dataframe
df_sanctions["year"] = df_sanctions["date"].dt.year

# Number of sanctions and mise en demeure per year
df_sanctions_year = df_sanctions.groupby(["year","nature_delib"])["doc_id"].nunique().reset_index(name="nb_decisions")
print("Number of decisions per year and type :")
print(df_sanctions_year)

# Rules (recommendations and standards) for the vector search
df_rules_all = df[df["nature_delib"].isin(["Recommandation/Lignes directrices","Référentiel/Règlement type/Norme"]) & (df["date"] >= "2018-05-25")].copy()
print("Number of rules:", df_rules_all["doc_id"].nunique())

df_rules_all["year"] = df_rules_all["date"].dt.year



Number of sanctions and mise en demeure: 160
Minimal date : 2018-06-21 00:00:00
Maximal date : 2026-11-12 00:00:00
Number of decisions per year and type :
    year     nature_delib  nb_decisions
0   2018  Mise en demeure            22
1   2018         Sanction             7
2   2019  Mise en demeure             6
3   2019         Sanction             5
4   2020  Mise en demeure             4
5   2020         Sanction            12
6   2021  Mise en demeure             4
7   2021         Sanction            14
8   2022  Mise en demeure            24
9   2022         Sanction            16
10  2023  Mise en demeure             1
11  2023         Sanction            18
12  2024         Sanction             8
13  2025         Sanction            10
14  2026         Sanction             9
Number of rules: 84


In [55]:
# Rules to keep for PME 

rules_to_keep = [
    "2020-091", "2020-092", "2025-131", "2026-042", "2026-114",
    "2019-160", "2022-126", "2026-031", "2026-054", "2023-064", "2019-001", "2025-028",
    "2022-100", "2025-019", "2021-122", "2023-050", "2026-022",
    "2018-326", "2018-327", "2019-011", "2019-118",
    "2021-131", "2021-130", "2018-303", "2021-057",
    "2025-024", "2024-011", "2025-010", "2025-041", "2025-047", "2024-041",
    "2026-063", "2021-043",
    "2020-081", "2022-067", "2020-076",
    "2021-070", "2026-045",
]

df_rules = df_rules_all[df_rules_all["number"].isin(rules_to_keep)].copy()

print("Number of rules after filtering for PME:", df_rules["doc_id"].nunique()) # 2 are missing

# Check if all rules to keep are present in the dataframe
# Note: some numbers had trailing spaces (e.g. '2026-022 '), fixed with .str.strip()
keep = set(rules_to_keep)
end = set(df_rules["number"].unique())
print("Rules to keep but not found in the dataframe:", keep - end)

found = df[df["full_title"].str.contains("2026-022")]
print(found["number"].unique())

found = df[df["full_title"].str.contains("2026-063")]
print(found["number"].unique())



Number of rules after filtering for PME: 38
Rules to keep but not found in the dataframe: set()
<ArrowStringArray>
['2026-022', 'DR-2026-022']
Length: 2, dtype: str
<ArrowStringArray>
['2026-063', 'DR-2026-063']
Length: 2, dtype: str


In [56]:
# Empty value
print("Empty values in sanctions dataframe:")
print(df_sanctions.isna().sum())

print("Empty values in rules dataframe:")
print(df_rules.isna().sum())

# Status , Nature ? 
print(df_sanctions["nature"].value_counts())
print(df_sanctions["status"].value_counts())

# Drop columns 
cols_to_drop = ["chunk_id", "chunk_xxh64", "nature", "status", "chunk_text", "embeddings_bge-m3"]

df_sanctions = df_sanctions.drop(columns=cols_to_drop)
df_rules = df_rules.drop(columns=cols_to_drop)

print("Columns in sanctions dataframe:", df_sanctions.columns)
print("Columns in rules dataframe:", df_rules.columns)

Empty values in sanctions dataframe:
chunk_id             0
doc_id               0
chunk_index          0
chunk_xxh64          0
nature               0
status               0
nature_delib         0
title                0
full_title           0
number               0
date                 0
text                 0
chunk_text           0
embeddings_bge-m3    0
year                 0
dtype: int64
Empty values in rules dataframe:
chunk_id             0
doc_id               0
chunk_index          0
chunk_xxh64          0
nature               0
status               0
nature_delib         0
title                0
full_title           0
number               0
date                 0
text                 0
chunk_text           0
embeddings_bge-m3    0
year                 0
dtype: int64
nature
DELIBERATION    4841
DECISION         387
Name: count, dtype: int64
status
VIGUEUR    5228
Name: count, dtype: int64
Columns in sanctions dataframe: Index(['doc_id', 'chunk_index', 'nature_delib', 'title', '

# Etape 1 (suite) : regroupement des chunks par décision

Une ligne par décision (sanctions et mises en demeure) pour l'extraction par le LLM


In [57]:
# Sort chunks by position, then merge them into one row per decision
df_sanctions_sorted = df_sanctions.sort_values(by=["doc_id", "chunk_index"])
df_sanctions_grouped = df_sanctions_sorted.groupby(["doc_id"]).agg({"nature_delib": "first", "title":"first","full_title":"first","number":"first","date": "first", "text": " ".join,"year":"first"}).reset_index()

print("The size of df_sanctions_grouped is :", len(df_sanctions_grouped))

print("Five first lines of df_sanctions_grouped are :")
print(df_sanctions_grouped.head(5))

print("Last words of the first decision text :") # The fine amount is present; company names are anonymized
print(df_sanctions_grouped["text"].iloc[0][-1000:])



The size of df_sanctions_grouped is : 160
Five first lines of df_sanctions_grouped are :
                 doc_id     nature_delib  \
0  CNILTEXT000037110598         Sanction   
1  CNILTEXT000037217051  Mise en demeure   
2  CNILTEXT000037217090  Mise en demeure   
3  CNILTEXT000037217124  Mise en demeure   
4  CNILTEXT000037217191  Mise en demeure   

                                       title  \
0  Délibération SAN-2018-003 du 21 juin 2018   
1      Décision MED-2018-022 du 25 juin 2018   
2    Délibération 2018-287 du 5 juillet 2018   
3      Décision MED-2018-023 du 25 juin 2018   
4    Délibération 2018-288 du 5 juillet 2018   

                                          full_title        number       date  \
0  Délibération de la formation restreinte n° SAN...  SAN-2018-003 2018-06-21   
1  Décision n° MED 2018-022 du 25 juin 2018 metta...  MED-2018-022 2018-06-25   
2  Délibération du bureau de la Commission nation...      2018-287 2018-07-05   
3  Décision n° MED 2018-023 du 25

In [58]:
# Size of the text
len_text = df_sanctions_grouped["text"].str.len()
print("Size of the text :")
print(len_text)

print("Max size of the text :", len_text.max())
print("Min size of the text :", len_text.min())
print("Mean size of the text :", len_text.mean())

print("Number of decisions with text size < 1000 :", (len_text < 1000).sum())
print("Decisions with text size < 1000 :")
print(df_sanctions_grouped[df_sanctions_grouped["text"].str.len() < 1000])

short = df_sanctions_grouped[len_text < 1000]
print(short["text"].iloc[0])

Size of the text :
0       25955
1       28573
2        5084
3       23168
4        5055
        ...  
155    190417
156     98859
157     56609
158     59507
159     59862
Name: text, Length: 160, dtype: int64
Max size of the text : 190417
Min size of the text : 913
Mean size of the text : 38441.5875
Number of decisions with text size < 1000 : 2
Decisions with text size < 1000 :
                  doc_id     nature_delib  \
94  CNILTEXT000045835124  Mise en demeure   
98  CNILTEXT000045835356  Mise en demeure   

                                     title  \
94  Décision MED-2022-053 du 25 avril 2022   
98  Décision MED-2022-050 du 25 avril 2022   

                                           full_title        number  \
94  Clôture de la décision n°MED-2022-053 du 25 av...  MED-2022-053   
98  Décision n°MED-2022-050 du 25 avril 2022 metta...  MED-2022-050   

         date                                               text  year  
94 2022-04-25  La Présidente\nCOMMUNE DE X\n HÔTEL DE V

In [59]:
# Remove closure letters (no useful content)
df_sanctions_grouped = df_sanctions_grouped[~df_sanctions_grouped["text"].str.contains("clôture du dossier")]

print(len(df_sanctions_grouped), "decisions after removing closure letters")

158 decisions after removing closure letters


In [60]:
# Anonymized companies ? 

df_2025 = df_sanctions_grouped[df_sanctions_grouped["year"] == 2025]
text_2025 = df_2025["text"].iloc[0]   
print("Text of the 2025 decision :")
print(text_2025[-1000:])  # Print the last 1000 characters of the text

# Company names are visible in recent decisions, but anonymized ("X") after 2 years

Text of the 2025 decision :
tions électroniques et 7 du règlement (UE) 2016/679 du Parlement européen et du Conseil du 27 avril 2016 ; 
• prononcer à l’encontre de la société SOLOCAL MARKETING SERVICES une injonction de cesser de procéder à des opérations de prospection commerciale par voie électronique en l’absence d’un consentement valable ; • assortir l’injonction d’une astreinte de dix mille (10 000) euros par jour de retard à l’issue d’un délai de neuf (9) mois suivant la notification de la présente délibération, les justificatifs de la mise en conformité devant être adressés à la formation restreinte dans ce délai ; 
• rendre publique, sur le site web de la CNIL et sur le site web de Légifrance, sa délibération, qui ne permettra plus d’identifier nommément la société à l’issue d’une durée de deux ans à compter de sa publication.
Le président
Philippe-Pierre CABOURDIN
Cette décision est susceptible de faire l’objet d’un recours devant le Conseil d’État dans un délai de deux mois à

In [61]:
# df_rules 

print(df_rules["full_title"].nunique())

df_rules_sorted = df_rules.sort_values(by=["doc_id", "chunk_index"])
df_rules_grouped = df_rules_sorted.groupby(["doc_id"]).agg({"nature_delib": "first", "title":"first","full_title":"first","number":"first","date": "first", "text": " ".join,"year":"first"}).reset_index()

# Affiche tous les titres de règles 

for title in df_rules_grouped["full_title"]:
    print(title)

# empty title (blank lines in the list came from titles ending with "\n", now stripped)
empty_title = df_rules_all[df_rules_all["full_title"] == ""]
print("Empty title in rules dataframe :")
print(empty_title)


38
Délibération n° 2018-303 du 06 septembre 2018 portant adoption d'une recommandation concernant le traitement des données relatives à la carte de paiement en matière de vente de biens ou de fourniture de services à distance et abrogeant la délibération n° 2017-222 du 20 juillet 2017
Délibération n° 2018-327 du 11 octobre 2018 portant adoption de la liste des types d’opérations de traitement pour lesquelles une analyse d’impact relative à la protection des données est requise
Délibération n° 2018-326 du 11 octobre 2018 portant adoption de lignes directrices sur les analyses d'impact relatives à la protection des données (AIPD) prévues par le règlement général sur la protection des données (RGPD)
Délibération n° 2019-011 du 31 janvier 2019 modifiant la délibération n° 2018-327 du 11 octobre 2018 portant adoption de la liste des types d'opérations de traitement pour lesquelles une analyse d'impact relative à la protection des données est requise
Délibération n° 2019-118 du 12 septembre 

In [62]:
# Chunk text size in rules dataframe

len_text_rules = df_rules_grouped["text"].str.len()
print("Size of the text in rules dataframe :")
print(len_text_rules)

print("Max size of the text in rules dataframe :", len_text_rules.max())
print("Min size of the text in rules dataframe :", len_text_rules.min())
print("Mean size of the text in rules dataframe :", len_text_rules.mean())




Size of the text in rules dataframe :
0     30148
1      6697
2     13422
3      2273
4      5998
5     46231
6      1671
7     34017
8     26647
9     28474
10    27985
11    46923
12    48772
13    21734
14    17736
15    26423
16    45130
17    41416
18    33238
19    44767
20    95420
21    56563
22     2270
23     4059
24     1699
25     2188
26     4398
27     2115
28     1864
29     1677
30    13078
31     2256
32     1004
33     2319
34     2517
35     1818
36     2519
37     2173
Name: text, dtype: int64
Max size of the text in rules dataframe : 95420
Min size of the text in rules dataframe : 1004
Mean size of the text in rules dataframe : 19727.342105263157


In [63]:
# size 2000 little : strange 
print("Number of rules with text size < 5000 :", (len_text_rules < 5000).sum())
rules_5000 = df_rules_grouped[df_rules_grouped["text"].str.len() < 5000]
print("Rules with text size < 5000 :")
print(rules_5000["text"].iloc[0])

for number, text in zip(rules_5000["number"], rules_5000["text"]):
    print(number, "-", len(text), "characters")
    print(text[-300:])
    print("-" * 50)

# useless chunk (no useful content)

Number of rules with text size < 5000 : 17
Rules with text size < 5000 :
La Commission nationale de l'informatique et des libertés,
Vu la convention n° 108 du Conseil de l'Europe pour la protection des personnes à l'égard du traitement automatisé des données à caractère personnel ;
Vu le règlement (UE) 2016/679 du Parlement européen et du Conseil du 27 avril 2016 relatif à la protection des personnes physiques à l'égard du traitement des données à caractère personnel et à la libre circulation de ces données, et abrogeant la directive 95/46/CE, notamment son article 35 ;
Vu la loi n° 78-17 du 6 janvier 1978 modifiée relative à l'informatique, aux fichiers et aux libertés ;
Vu le décret n° 2005-1309 du 20 octobre 2005 modifié pris pour l'application de la loi n° 78-17 du 6 janvier 1978 relative à l'informatique, aux fichiers et aux libertés ;
Vu la délibération n° 2018-327 du 11 octobre 2018 portant adoption de la liste des types d'opérations de traitement pour lesquelles une analyse d'i

In [64]:
# Rules without content: annex missing from the dataset, or simple modifications
rules_without_content = [
    "2019-011", "2020-076", "2024-011", "2024-041", "2025-010",
    "2025-019", "2025-024", "2025-028", "2025-041", "2025-047",
    "2026-022", "2026-031", "2026-042", "2026-045", "2026-054",
    "2026-063", "2026-114",
]

df_rules = df_rules[~df_rules["number"].isin(rules_without_content)].copy()
print("Number of rules with content:", df_rules["doc_id"].nunique())


Number of rules with content: 21


In [65]:
# Chunk size in rules (chunks, not full documents) /  checking if some chunks are very short (e.g. just a signature or the end of a document), we want clean chunks for the vector search
len_chunks_rules = df_rules["text"].str.len()
print("Rules chunks - min:", len_chunks_rules.min())
print("Rules chunks - max:", len_chunks_rules.max())
print("Rules chunks - mean:", round(len_chunks_rules.mean()))

# Very short chunks: often just a signature or the end of a document
short_chunks = df_rules[len_chunks_rules < 100]
print("Number of chunks < 100 characters:", len(short_chunks))

for number, text in zip(short_chunks["number"], short_chunks["text"]):
    print(number, "-", repr(text))
    print("-" * 50)

# Same for sanctions dataframe
len_chunks_sanctions = df_sanctions["text"].str.len()
print("Sanctions chunks - min:", len_chunks_sanctions.min())
print("Sanctions chunks - max:", len_chunks_sanctions.max())
print("Sanctions chunks - mean:", round(len_chunks_sanctions.mean()))

short_chunks_sanctions = df_sanctions[len_chunks_sanctions < 100]
print("Number of sanctions chunks < 100 characters:", len(short_chunks_sanctions))

for number, text in zip(short_chunks_sanctions["number"], short_chunks_sanctions["text"]):
    print(number, "-", repr(text))
    print("-" * 50)

# Remove very short chunks (sentence fragments with no meaning on their own)
df_sanctions = df_sanctions[df_sanctions["text"].str.len() >= 100].copy()

# Remove closure letters: keep only decisions still in df_sanctions_grouped
df_sanctions = df_sanctions[df_sanctions["doc_id"].isin(df_sanctions_grouped["doc_id"])].copy()

print("Number of chunks:", len(df_sanctions))
print("Number of decisions:", df_sanctions["doc_id"].nunique())

Rules chunks - min: 37
Rules chunks - max: 1499
Rules chunks - mean: 1287
Number of chunks < 100 characters: 2
2021-070 - 'Article 3 - L’entrée en relation contractuelle entre la personne concernée et le mandataire'
--------------------------------------------------
2023-050 - 'l’ouverture est effectivement prévue.'
--------------------------------------------------
Sanctions chunks - min: 1
Sanctions chunks - max: 1499
Sanctions chunks - mean: 1176
Number of sanctions chunks < 100 characters: 50
SAN-2019-006 - '4. Sur la sanction et la publicité'
--------------------------------------------------
SAN-2019-006 - '65. L’article 83 du RGPD prévoit :'
--------------------------------------------------
SAN-2019-007 - 'personnelles contenues dans ses systèmes d’information, malgré la nature des données traitées.'
--------------------------------------------------
SAN-2020-003 - 'commerciales à ses prospects que pendant une période de deux ans.'
----------------------------------------------

In [66]:
# Save the cleaned data for the next steps
df_sanctions_grouped.to_parquet("../data/sanctions_decisions.parquet", index=False)
df_sanctions.to_parquet("../data/sanctions_chunks.parquet", index=False)
df_rules.to_parquet("../data/rules_chunks.parquet", index=False)

In [67]:
# Check: reload the files
check = pd.read_parquet("../data/sanctions_decisions.parquet")
print(len(check), "decisions")

check_chunks = pd.read_parquet("../data/sanctions_chunks.parquet")
print(len(check_chunks), "sanction chunks")

check_rules = pd.read_parquet("../data/rules_chunks.parquet")
print(check_rules["doc_id"].nunique(), "rules")


158 decisions
5176 sanction chunks
21 rules
